In [3]:
from ngsolve import *
from ngsolve.webgui import Draw
import math
import scipy.linalg
import numpy as np
from ngsolve.eigenvalues import SOAR
nr_eigs = 200
omega=Parameter(5)
from netgen.occ import *
from netgen.webgui import Draw as DrawGeo

a = 1
r = 0.38

rect = WorkPlane().RectangleC(a,a).Face()
circ = WorkPlane().Circle(0,0, r).Face()
# r2 = WorkPlane().Rotate(30).RectangleC(a, a/10).Face()
# circ += r2

outer = rect-circ
inner = rect*circ

outer.faces.name = "outer"
outer.faces.col=(1,1,0)

inner.faces.col=(1,0,0)
inner.faces.name="inner"
shape = Glue([outer, inner])

shape.edges.Max(X).name = "right"
shape.edges.Max(-X).name = "left"
shape.edges.Max(Y).name = "top"
shape.edges.Max(-Y).name = "bot"

shape.edges.Max(Y).Identify(shape.edges.Min(Y), "bt")
shape.edges.Max(X).Identify(shape.edges.Min(X), "lr")

mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.1))
mesh.Curve(5)
Draw (mesh)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [4]:
fes = Compress(Periodic(H1(mesh, complex=True, order=5)))
# fes = Periodic(H1(mesh, complex=True, order=4))

print (fes.ndof)
u,v = fes.TnT()

cf_mu = 1
cf_eps = mesh.MaterialCF({"inner":9}, default=1)

a = BilinearForm( 1/cf_mu*grad(u)*grad(v)*dx-cf_eps*omega**2*u*v*dx )
b = BilinearForm( 1/cf_mu*(u*grad(v)[0]-grad(u)[0]*v)*dx )
c = BilinearForm( -1/cf_mu*u*v*dx )
a1 = BilinearForm( 1/cf_mu*grad(u)*grad(v)*dx )
a2 = BilinearForm( cf_eps*u*v*dx )
# a = a1 - omega**2 * a2


a.Assemble()
b.Assemble()
c.Assemble()
a1.Assemble()
a2.Assemble()

inv = a.mat.Inverse(freedofs=fes.FreeDofs(), inverse="sparsecholesky")
M1 = -inv@b.mat
M2 = -inv@c.mat
Q = SOAR(M1,M2, nr_eigs)

len(Q)


2550


200

In [5]:
def SolveProjectedSmall (Am, Bm, Cm):
    half = Am.h
    Cmi = Cm.I
    K = Matrix(2*half, 2*half, complex = fes.is_complex)

    K[:half, :half] = -Cmi*Bm
    K[:half, half:] = -Cmi*Am
    K[half:, :half] = Matrix(half, complex=True).Identity()
    K[half:, half:] = 0

    Kr = Matrix(2*half)
    Kr.A = K.A.real
    lam, eig = scipy.linalg.eig(Kr)
    vecs = Matrix(eig)[0:len(Q),:]
    lam = Vector(lam)

    return lam, vecs

def SolveProjected(mata, matb, matc, Q):
    Am = InnerProduct(Q, mata*Q, conjugate = True)
    Bm = InnerProduct(Q, matb*Q, conjugate = True)
    Cm = InnerProduct(Q, matc*Q, conjugate = True)
    return SolveProjectedSmall (Am, Bm, Cm)

lams, vecs = SolveProjected(a.mat, b.mat, c.mat, Q)
Z = (Q * vecs).Evaluate()
for vec in Z: vec /= Norm(vec)

gfu = GridFunction(fes)
ind = np.absolute(np.asarray(lams)-6j).argmin()
gfu.vec.data = Z[ind]
Draw (gfu, animate_complex=False, deformation=True, scale=2)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene